# Cody, Kumar & Tzounopoulos (2024) on dandiset 001256

**Paper.** Cody P, Kumar M, Tzounopoulos T (2024). Cortical zinc signaling is necessary for changes in
mouse pupil diameter that are evoked by background sounds with different contrasts. J Neurosci 44(11)
e0939232024, doi:10.1523/JNEUROSCI.0939-23.2024.

**Dataset.** DANDI:001256, version 0.241120.2150 (doi:10.48324/dandi.001256/0.241120.2150), streamed from
the archive. The paper cites no version (it names no dandiset); 0.241120.2150 is the only published
version and the most recent, and the draft has the same 30 assets and size, so it is pinned.

**Claims.** `claims/001256/10.1523-jneurosci.0939-23.2024.yaml`, written from the version of record
(main text, Methods and figure legends; the paper has no supplement and links no code).

| Claim | Tested |
|---|---|
| fig1f-tone-response-inverse-u | tested |
| fig1g-reliability-intermediate | tested |
| fig1h-cgc-largest-intermediate | tested |
| fig1h-cgc-evident-all-bins | tested |
| fig1i-baseline-contrast-interaction | tested |
| fig1i-contrast-effect-intermediate-only | tested |
| fig2b-evoked-pd-whole-drc | tested |
| fig2c-evoked-pd-1-2s | tested; blocked (readings disagree, as the criterion says) |
| fig2e-drc-responses-no-difference | tested |
| fig3b-lme-coefficients | tested |
| fig3c-lme-likelihood-ratio | tested |
| fig4c-wt-contrast-zx1-effects | tested |
| fig4-wt-posthoc | tested |
| fig4f-ko-no-contrast-effect | tested; blocked (readings disagree, as the criterion says) |
| fig4f-ko-zx1-slope | tested; blocked (depends on the claim above) |
| fig5-contralateral-zx1-increases | blocked, data missing (no widefield imaging in the dandiset) |
| fig5-ipsilateral-zx1-reduces | blocked, data missing |
| fig5c-no-dye-in-other-hemisphere | blocked, data missing |
| fig6a-baseline-pd-drift | tested |
| fig6b-acsf-contrast-effect | tested |
| fig6c-no-baseline-evoked-correlation | tested |
| methods-baseline-evoked-correlation | tested |
| fig6d-ko-reduced-high-contrast | tested |
| discussion-zx1-wt-like-ko | tested |

Where a method detail is not in the claims file it is taken from the paper's Methods, cited in the helper
module `zinc_pupil.py`. No random numbers are used. Loading takes about one to two minutes.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import zinc_pupil as zp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("dandiset", zp.DANDISET_ID, "version", zp.DANDISET_VERSION)

In [ ]:
assets = zp.list_assets()
meta, trials = zp.load_all(assets)
print(f"{len(assets)} files, {len(trials)} trials read in {time.time() - T0:.0f} s")

## Data check

What was loaded, by subject, with the analysis labels the files carry in `session_description`.
The groups of the claims file are taken from these labels (A: `pupilPTinContrast`; C:
`pupil_contrast_ZnT3WT`; D: `pupil_contrast_ZnT3KO`; F: `pupil_contrast_ACSF`), and group B (Fig. 2d, e,
cells before and after ZX1) as the mice with CGC-assay trials both before and after ZX1, which no label
names.

In [ ]:
meta["labels"] = meta.session_description.str.split(" | ")
lab = meta.set_index("subject").labels
def in_label(name):
    return sorted(s for s, l in lab.items() if name in l)

GROUPS = {
    "A": in_label("pupilPTinContrast"),
    "C": in_label("pupil_contrast_ZnT3WT"),
    "D": in_label("pupil_contrast_ZnT3KO"),
    "F": in_label("pupil_contrast_ACSF"),
    "ZX1_drift": in_label("ZX1_uBasePupilPctMax_wEpoch"),
    "ACSF_drift": in_label("ACSF_uBasePupilPctMax_wEpoch"),
    "base_vs_evoked": in_label("pupilBaseline_vs_evoked"),
}
trials["treat"] = trials.treatment.map({"none": "CTRL", "preZX1": "CTRL", "preACSF": "CTRL",
                                        "postZX1": "ZX1", "postACSF": "ACSF"})
pt_tab = pd.crosstab(trials[trials.kind == "pt"].subject, trials[trials.kind == "pt"].treat)
GROUPS["B"] = sorted(pt_tab.index[(pt_tab.get("CTRL", 0) > 0) & (pt_tab.get("ZX1", 0) > 0)])
summary = meta[["subject", "genotype", "sex", "age"]].set_index("subject")
for g, subs in GROUPS.items():
    summary[g] = summary.index.isin(subs).astype(int)
print(summary.sort_index().to_string())
for g, subs in GROUPS.items():
    sx = summary.loc[subs, "sex"].value_counts().to_dict()
    gt = summary.loc[subs, "genotype"].value_counts().to_dict()
    print(f"group {g}: {len(subs)} mice, sex {sx}, genotype {gt}")
print()
print("trials by kind and treatment")
print(pd.crosstab(trials.kind, trials.treat))
print()
print("trials by contrast of the first DRC block (kind pt and drc)")
print(pd.crosstab(trials[trials.kind.isin(["pt", "drc"])].kind, trials.contrast))
print("stimDelay values (s):", trials.groupby("kind").stimDelay.unique().to_dict())
print("2P frame rates:", trials.frameRate.unique(), "frames per trial:", trials.groupby("kind").nFrames.unique().to_dict())
st = trials.starting_time.astype(float)
ft = pd.to_datetime(trials.fileTimeInstantiate)
off = (ft - ft.groupby(trials.subject).transform(lambda x: x[st.loc[x.index] == 0].min())).dt.total_seconds()
print("trials whose table starting_time differs from their .tif creation time (relative to the reference .tif) by more than 1 s:",
      int(((st - off).abs() > 1).sum()), "; by subject:", trials[(st - off).abs() > 1].subject.value_counts().to_dict())
print("  of these, starting_time - (creation time offset) in hours:", sorted(((st - off)[(st - off).abs() > 1] / 3600).round(2).unique().tolist())[:5])
print("pure-tone frequencies of the CGC assay:", trials[trials.kind == "pt"].pulseSets.str.extract(r"(\d+Hz)")[0].value_counts().to_dict())

Checks of the metadata against the values:

- Pupil series: two video frames per two-photon frame (10 and 5 Hz) is expected. Pupil series whose
  length does not match their trial (a series of a mapping trial's length attached to a DRC trial)
  are counted and left out. Every pupil series's description names `TwoPhotonSeries_036` (or the
  file's last series); the series is matched to its trial by the .tif file named in its `comments`.
- Units: the pupil radius is in "na" (pixels, judged from the values) and the fluorescence in
  "lumens" (raw intensity); both analyses use ratios, so the units do not enter.
- Raw pupil traces hold dropped frames (NaN) and tracking failures.

In [ ]:
snd = trials[trials.kind.isin(["pt", "drc"])].copy()
snd["has_pupil"] = snd.pupil.notna()
snd["len_ok"] = [p is not None and len(p) == 2 * int(n) for p, n in zip(snd.pupil, snd.nFrames)]
print("sound trials:", len(snd), "with pupil:", snd.has_pupil.sum(), "pupil length = 2 x frames:", snd.len_ok.sum())
print("mismatched pupil lengths by subject:", snd[snd.has_pupil & ~snd.len_ok].subject.value_counts().to_dict())
raw = snd[snd.len_ok]
nan_frac = raw.pupil.map(lambda p: np.isnan(p).mean())
print(f"fraction of pupil frames missing per trial: median {nan_frac.median():.3f}, "
      f"trials all missing {int((nan_frac == 1).sum())}")
rmax = raw.pupil.map(lambda p: np.nanmax(p) if np.isfinite(p).any() else np.nan)
rmed = raw.pupil.map(lambda p: np.nanmedian(p) if np.isfinite(p).any() else np.nan)
print(f"raw radius: median of trial medians {rmed.median():.1f}, range of trial medians "
      f"{rmed.min():.1f} to {rmed.max():.1f}; trials whose maximum exceeds 3 x their median: "
      f"{int((rmax > 3 * rmed).sum())} (largest raw value {np.nanmax(rmax):.3g})")

# Pupil cleaning: the paper's single pass, and the pass repeated with a spike filter (main).
single = zp.pupil_measures(trials, max_pass=1, spike_filter=False, max_missing=1.0)
tr = zp.pupil_measures(trials)
print(f"trials with a usable pupil: single pass {len(single)}, main cleaning {len(tr)}")
amax = pd.DataFrame({
    "max_single_pass": single.groupby("subject").pclean.apply(lambda s: max(np.max(x) for x in s)),
    "max_main": tr.groupby("subject").pclean.apply(lambda s: max(np.max(x) for x in s)),
    "median_main": tr.groupby("subject").pclean.apply(lambda s: np.median(np.concatenate(list(s)))),
    "trials_main": tr.groupby("subject").size(),
})
print(amax.round(1).to_string())
print("baseline PD (% animal max), main cleaning: range", tr.base_pct_max.min().round(1), "to",
      tr.base_pct_max.max().round(1), "; below 50%:", int((tr.base_pct_max < 50).sum()), "of", len(tr))

With one pass, several animals keep pupil "maxima" of 200 to 790000 pixels, against medians of 20 to 100,
so a baseline in % of the animal's maximum would be near 0 for every trial of those animals. The main
cleaning (Hampel spike filter, the derivative step repeated, and trials with more than half of their frames
missing dropped) is an assumption; the paper's own DeepLabCut points and likelihoods, which would let
the frames be dropped as it describes, are not in the files.

Clock check. The pupil series start at their trial's start time (asserted on loading). Sound onset is
`stimDelay` after the first frame. If this is right, the population calcium response should rise at the
pure tone (2 s after onset, in both contrasts) and not at other times, and the pupil should dilate
after onset, not before. Shifting the assumed onset by whole seconds moves the tone response away from
2 s.

In [ ]:
ptr = trials[trials.kind == "pt"]
pop = []
for _, r in ptr.iterrows():
    d, t = zp.dff(np.asarray(r.F, float), r.stimDelay)
    pop.append(np.nanmean(d, 1))
pop = np.nanmean(np.vstack(pop), 0)
t = zp.frame_times(len(pop), 4)
print("population mean dF/F (%) of CGC-assay trials by time from sound onset:")
print(pd.Series(pop.round(2), index=t.round(1)).loc[-1.2:4.0].to_string())
d1 = np.diff(pop)
k = np.argmax(d1[(t[:-1] >= 0.5)]) + np.sum(t[:-1] < 0.5)
print(f"largest frame-to-frame rise after 0.5 s: from {t[k]:.1f} to {t[k+1]:.1f} s (tone at 2.0 s)")
ev = tr[tr.kind.isin(["pt", "drc"])]
m = np.nanmean(np.vstack([x[:200] for x in ev.ev_trace]), 0)
tp = np.arange(200) / 10 - 4
print("mean pupil (% baseline) over trials: -4 to 0 s", m[(tp >= -4) & (tp < 0)].mean().round(2),
      "; 0 to 1 s", m[(tp >= 0) & (tp < 1)].mean().round(2), "; 1 to 2 s", m[(tp >= 1) & (tp < 2)].mean().round(2),
      "; 2 to 4 s", m[(tp >= 2) & (tp < 4)].mean().round(2))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(t, pop); ax[0].axvline(0, c="k", ls=":"); ax[0].axvline(2, c="g", ls=":")
ax[0].set(xlabel="time from sound onset (s)", ylabel="mean dF/F (%)", title="CGC assay, all cells")
ax[1].plot(tp, m); ax[1].axvline(0, c="k", ls=":")
ax[1].set(xlabel="time from sound onset (s)", ylabel="pupil (% baseline)", title="all sound trials")
plt.tight_layout(); plt.show()
print("plotted ranges: dF/F", pop.min().round(2), "to", pop.max().round(2), "; pupil", m.min().round(2), "to", m.max().round(2))

In [ ]:
# Recordings deposited twice: identical pupil traces or fluorescence across trials.
import hashlib
def h(a):
    a = np.asarray(a, float)
    return None if not np.isfinite(a).any() else hashlib.md5(np.nan_to_num(a).tobytes()).hexdigest()
hp = raw.pupil.map(h).dropna()
hf = ptr.F.map(h)
print("identical pupil traces (excluding all-missing):", int(hp.duplicated().sum()),
      "; identical fluorescence arrays:", int(hf.duplicated().sum()))
print("subjects:", meta.subject.nunique(), "files:", len(meta))

## Figure 1: pure-tone responses, reliability and contrast scaling across baseline PD (group A)

Group A is the ten mice the files label `pupilPTinContrast` (the paper gives eight, four F and four M; the
files give ten, four F and six M, one of them ZnT3-WT). CTRL trials of the CGC assay (DRC with the pure
tone) are used. Per trial: the pure-tone response trace (dF/F minus its 1-2 s mean), the peak in each
window, and the trial's baseline PD in % of the animal's maximum.

Readings run (claims file, shared methods): cell inclusion, SD over time of the trial-averaged trace
(`time`) or across trials (`trials`), and responsive in both contrasts (`both`) or with contrasts
pooled (`pooled`); peak window 2-2.8 s or 2-3.2 s; peak as the maximum of the cell's trial-averaged
trace in the bin (`avgmax`) or the mean of its trial maxima (`meanmax`); bin 1 all trials below 21%
(`open`) or 10 to 21% (`10-21`). The FRA-mapping reading of inclusion (claims file, (b)) cannot be run:
mapping trials are in 5 of 30 files and only one group A mouse (AA0318).

In [ ]:
A = GROUPS["A"]
pt_ctrl = trials[(trials.kind == "pt") & (trials.treat == "CTRL") & trials.subject.isin(A)]
ct = zp.cell_trial_table(pt_ctrl)
base = tr.set_index("trial_id").base_pct_max
ct["base"] = ct.trial.map(base)
print("group A CTRL CGC-assay trials:", pt_ctrl.trial_id.nunique(), "; with a usable pupil:",
      int(pt_ctrl.trial_id.isin(base.index).sum()))
print("cells (ROIs) in group A:", ct.cell.nunique(), "; by mouse:", ct.groupby("subject").cell.nunique().to_dict())
INCL = {}
for sd in ("time", "trials"):
    for both in (True, False):
        INCL[(sd, "both" if both else "pooled")] = zp.responsive(ct, (2.0, 2.8), sd=sd, by_contrast=both)
print("tone-responsive cells under each inclusion reading (paper: 295 from eight mice):")
for k, v in INCL.items():
    print(f"  SD {k[0]:6s} {k[1]:6s}: {len(v)} cells")
ctb = ct[ct.base.notna()].copy()
print("cell-trials without a usable pupil (dropped):", int(ct.base.isna().sum()), "of", len(ct))

def bin_values(d, peak_col, window, mode, lower, by=("cell", "bin")):
    """Average peak per cell and bin (and contrast when in `by`)."""
    d = d.copy()
    d["bin"] = zp.pd_bin(d.base.values, 10 if lower == "10-21" else None)
    d = d[d.bin >= 0]
    out = []
    for key, g in d.groupby(list(by)):
        if mode == "meanmax":
            v = g[peak_col].mean()
        else:
            tr_ = np.vstack(g.trace.values).mean(0)
            t_ = g.t.iloc[0]
            v = tr_[(t_ >= window[0]) & (t_ < window[1])].max()
        out.append(dict(zip(by, key if isinstance(key, tuple) else (key,)), value=v, n=len(g)))
    return pd.DataFrame(out)

def kw(df):
    groups = [g.value.values for _, g in df.groupby("bin")]
    H, p = stats.kruskal(*groups)
    means = df.groupby("bin").value.mean()
    return H, p, int(means.idxmax()), means

WINDOWS = {"2-2.8": (2.0, 2.8), "2-3.2": (2.0, 3.2)}
rows = []
for (sd, bc), cells in INCL.items():
    d0 = ctb[ctb.cell.isin(cells)]
    for wn, w in WINDOWS.items():
        for mode in ("avgmax", "meanmax"):
            for lower in ("open", "10-21"):
                v = bin_values(d0, f"peak_{w[0]}_{w[1]}", w, mode, lower)
                H, p, top, means = kw(v)
                rows.append({"SD": sd, "contrasts": bc, "window": wn, "peak": mode, "bin1": lower,
                             "cells": v.cell.nunique(), "cell_bins": len(v), "H": round(H, 2), "p": p,
                             "top_bin": zp.BIN_LABELS[top], "inverse_U": 0 < top < 6,
                             **{f"m{b}": round(means.get(b, np.nan), 2) for b in range(7)}})
F1F = pd.DataFrame(rows)
print(F1F.to_string())
print("readings with p < 0.05 and an interior top bin:", int(((F1F.p < 0.05) & F1F.inverse_U).sum()), "of", len(F1F))

In [ ]:
# Main reading (first listed): SD over time, both contrasts, 2-2.8 s, avgmax, bin 1 open; by mouse.
main_cells = INCL[("time", "both")]
v = bin_values(ctb[ctb.cell.isin(main_cells)], "peak_2.0_2.8", (2.0, 2.8), "avgmax", "open")
v["subject"] = v.cell.str.split("/").str[0]
print("cell-bins per bin:", v.groupby("bin").size().to_dict())
print("mean peak (% dF/F) by mouse and bin:")
print(v.pivot_table(index="subject", columns="bin", values="value", aggfunc="mean").round(1).to_string())
print("cells and bins per mouse:", v.groupby("subject").cell.nunique().to_dict())
fig, ax = plt.subplots(figsize=(4, 3))
mm = v.groupby("bin").value.agg(["mean", "sem"])
ax.errorbar([zp.BIN_LABELS[i] for i in mm.index], mm["mean"], mm["sem"], marker="o")
ax.set(xlabel="baseline PD (% animal max)", ylabel="peak pure-tone dF/F (%)"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print("plotted means range", mm["mean"].min().round(2), "to", mm["mean"].max().round(2))

In [ ]:
print("trials with baseline PD below 10% of animal max:", int((ctb.drop_duplicates("trial").base < 10).sum()),
      "; so the two bin-1 readings are the same here")

**Result: reproduced.** Under the main reading (SD over time, both contrasts, 2-2.8 s, maximum of the averaged trace) 318 cells are tone-responsive, within the tolerance of 295 (265.5 to 324.5); the other inclusion readings give 339, 215 and 241. Peak responses differ across the seven baseline PD bins (Kruskal-Wallis H = 35.69, p = 3.2e-06), with the largest mean in the 32-43% bin (15.15% dF/F) against 8.12 in the lowest bin and 13.08 in the highest. All 32 readings give p < 0.05 (2.2e-07 to 1.4e-04) and the largest mean in the 32-43% bin. The bins are not independent of mouse: the lowest bin holds cells of three mice only (AA0335, AA0350, AA0353) and the highest of five.

### Reliability (Fig. 1g)

Per cell and bin, the mean over all pairs of the cell's trials in the bin of the zero-lag correlation of
the pure-tone response traces at 2 to 3.2 s (six frames at 5 Hz): (a) Pearson, (b) normalized without
mean removal.

In [ ]:
def reliability(d, how):
    d = d.copy()
    d["bin"] = zp.pd_bin(d.base.values)
    out = []
    for (cell, b), g in d.groupby(["cell", "bin"]):
        if len(g) < 2:
            continue
        t_ = g.t.iloc[0]
        X = np.vstack(g.trace.values)[:, (t_ >= 2.0) & (t_ < 3.2)]
        if how == "pearson":
            X = X - X.mean(1, keepdims=True)
        nrm = np.linalg.norm(X, axis=1)
        ok = nrm > 0
        X = X[ok] / nrm[ok, None]
        if len(X) < 2:
            continue
        C = X @ X.T
        iu = np.triu_indices(len(X), 1)
        out.append({"cell": cell, "bin": b, "value": C[iu].mean(), "n": len(X)})
    return pd.DataFrame(out)

rows = []
for (sd, bc), cells in INCL.items():
    for how in ("pearson", "no_mean_removal"):
        v = reliability(ctb[ctb.cell.isin(cells)], how)
        H, p, top, means = kw(v)
        rows.append({"SD": sd, "contrasts": bc, "cc": how, "cells": v.cell.nunique(), "cell_bins": len(v),
                     "H": round(H, 2), "p": p, "top_bin": zp.BIN_LABELS[top], "interior": 0 < top < 6,
                     **{f"m{b}": round(means.get(b, np.nan), 3) for b in range(7)}})
F1G = pd.DataFrame(rows)
print(F1G.to_string())
print("cell-bins with fewer than two trials have no value")

**Result: reproduced.** Reliability differs across bins under all eight readings (Kruskal-Wallis p = 1.0e-08 to 9.3e-07), and the bin with the highest mean is 21-32% in every reading (Pearson, main inclusion: 0.766 against 0.633 in the lowest bin and 0.590 in the highest).

### Contrast scaling factor (Fig. 1h)

Per cell (responsive in both contrasts) and bin with trials of both contrasts: average peak in LOW
divided by average peak in HIGH, under both peak readings and windows.

In [ ]:
def scaling(d, w, mode):
    vals = bin_values(d, f"peak_{w[0]}_{w[1]}", w, mode, "open", by=("cell", "bin", "contrast"))
    pv = vals.pivot_table(index=["cell", "bin"], columns="contrast", values="value").dropna()
    pv["value"] = pv.LOW / pv.HIGH
    return pv.reset_index()

rows, SCAL = [], {}
for sd in ("time", "trials"):
    cells = INCL[(sd, "both")]
    for wn, w in WINDOWS.items():
        for mode in ("avgmax", "meanmax"):
            v = scaling(ctb[ctb.cell.isin(cells)], w, mode)
            SCAL[(sd, wn, mode)] = v
            H, p, top, means = kw(v)
            med = v.groupby("bin").value.median()
            rows.append({"SD": sd, "window": wn, "peak": mode, "cells": v.cell.nunique(), "cell_bins": len(v),
                         "H": round(H, 2), "p": p, "top_bin(mean)": zp.BIN_LABELS[top], "interior": 0 < top < 6,
                         "HIGH<=0": int((v.HIGH <= 0).sum()),
                         **{f"mean{b}": round(means.get(b, np.nan), 2) for b in range(7)},
                         **{f"med{b}": round(med.get(b, np.nan), 2) for b in range(7)}})
F1H = pd.DataFrame(rows)
print(F1H.to_string())
v = SCAL[("time", "2-2.8", "avgmax")]
print("main reading: range of factors", v.value.min().round(2), "to", v.value.max().round(2))

**Result: reproduced.** The contrast scaling factor differs across bins under all eight readings (Kruskal-Wallis p = 2.5e-17 to 1.1e-09), and the bin with the highest mean is 54-65% in seven readings and 43-54% in one, never the first or last. The means are pulled by a few cells whose HIGH peak is near or below 0 (factors from -109.5 to 37.8 in the main reading); the medians also peak in the 54-65% bin (1.34) in the main reading.

In [ ]:
rows = []
for key, v in SCAL.items():
    for b, g in v.groupby("bin"):
        x = g.value.values
        rows.append({"SD": key[0], "window": key[1], "peak": key[2], "bin": zp.BIN_LABELS[b], "n": len(x),
                     "mean": x.mean().round(3), "median": np.median(x).round(3),
                     "wilcoxon_2s": stats.wilcoxon(x - 1).pvalue,
                     "wilcoxon_1s": stats.wilcoxon(x - 1, alternative="greater").pvalue,
                     "t_2s": stats.ttest_1samp(x, 1).pvalue,
                     "t_1s": stats.ttest_1samp(x, 1, alternative="greater").pvalue})
F1HE = pd.DataFrame(rows)
F1HE["all_sig"] = (F1HE[["wilcoxon_2s", "wilcoxon_1s", "t_2s", "t_1s"]] < 0.05).all(1) & (F1HE["mean"] > 1)
print(F1HE.round(4).to_string())
print(F1HE.groupby(["SD", "window", "peak"]).all_sig.sum().rename("bins significant under all four tests (of 7)").to_string())
for test in ["wilcoxon_2s", "wilcoxon_1s", "t_2s", "t_1s"]:
    print(test, "bins with p < 0.05 and mean > 1, by reading:",
          F1HE.assign(ok=(F1HE[test] < 0.05) & (F1HE["mean"] > 1)).groupby(["SD", "window", "peak"]).ok.sum().to_dict())

**Result: not reproduced.** Under no reading are all seven bins significantly above 1. The 32-43% bin has a mean factor of 1.006 to 1.035 and p >= 0.17 under every reading and test, and the best readings have six bins significant under some of the tests (for example SD over time, 2-3.2 s, mean of trial maxima, under both Wilcoxon tests and the one-sided t test). Under the main reading the 43-54% bin has a mean of 0.619 (median 1.207) and five bins pass all four tests.

### Two-way ANOVA of peak responses by bin and contrast (Fig. 1i)

In [ ]:
import statsmodels.api as sm
def anova_1i(d, w, mode, pairing):
    vals = bin_values(d, f"peak_{w[0]}_{w[1]}", w, mode, "open", by=("cell", "bin", "contrast"))
    if pairing == "both":
        pv = vals.pivot_table(index=["cell", "bin"], columns="contrast", values="value").dropna()
        keep = set(pv.index)
        vals = vals[[k in keep for k in zip(vals.cell, vals.bin)]]
    m = smf.ols("value ~ C(bin) * C(contrast)", data=vals).fit()
    a = sm.stats.anova_lm(m, typ=2)
    return vals, m, a

rows, ANOVA = [], {}
for (sd, bc), cells in INCL.items():
    for wn, w in WINDOWS.items():
        for mode in ("avgmax", "meanmax"):
            for pairing in ("all", "both"):
                vals, m, a = anova_1i(ctb[ctb.cell.isin(cells)], w, mode, pairing)
                ANOVA[(sd, bc, wn, mode, pairing)] = (vals, m, a)
                rows.append({"SD": sd, "contrasts": bc, "window": wn, "peak": mode, "values_in": pairing,
                             "n_values": len(vals), "df_err": int(m.df_resid),
                             "F_bin": a.loc["C(bin)", "F"].round(2), "p_bin": a.loc["C(bin)", "PR(>F)"],
                             "F_int": a.loc["C(bin):C(contrast)", "F"].round(2),
                             "p_int": a.loc["C(bin):C(contrast)", "PR(>F)"],
                             "F_contrast": a.loc["C(contrast)", "F"].round(2), "p_contrast": a.loc["C(contrast)", "PR(>F)"]})
F1I = pd.DataFrame(rows)
F1I["count_ok"] = (F1I.n_values - 1885).abs() <= 188.5
print(F1I.to_string())
print("paper: 1885 values (error df 1871), F_bin = 10.74, F_int = 6.01")

**Result: reproduced.** The readings that keep only cell-bins with both contrasts give 1830 (SD over time, both contrasts) and 1932 values (contrasts pooled), within the tolerance of 1885. In these, the bin effect is F = 38.07 to 41.12 (p <= 1.0e-43) and the bin by contrast interaction F = 2.21 to 3.02 (p = 0.0061 to 0.040), so both are significant. Keeping every cell-bin-contrast gives 2204 values and the interaction F = 5.41 (p = 1.5e-05). The bin effect is larger than the paper's F = 10.74 in every reading.

In [ ]:
def posthoc(vals, m):
    mse = m.mse_resid; df = m.df_resid
    g = vals.groupby(["bin", "contrast"]).value.agg(["mean", "count"])
    rows = []
    for b in range(7):
        if (b, "LOW") not in g.index or (b, "HIGH") not in g.index:
            continue
        lo, hi = g.loc[(b, "LOW")], g.loc[(b, "HIGH")]
        tval = (lo["mean"] - hi["mean"]) / np.sqrt(mse * (1 / lo["count"] + 1 / hi["count"]))
        p = 2 * stats.t.sf(abs(tval), df)
        rows.append({"bin": zp.BIN_LABELS[b], "LOW": round(lo["mean"], 2), "HIGH": round(hi["mean"], 2),
                     "t": round(tval, 2), "p_raw": p, "p_bonf7": min(1, p * 7), "p_bonf91": min(1, p * 91)})
    return pd.DataFrame(rows)

rows = []
for key, (vals, m, a) in ANOVA.items():
    ph = posthoc(vals, m)
    for fam in ("p_bonf7", "p_bonf91"):
        sig = ph.loc[ph[fam] < 0.05, "bin"].tolist()
        rows.append({"SD": key[0], "contrasts": key[1], "window": key[2], "peak": key[3], "values_in": key[4],
                     "family": fam, "significant_bins": sig, "only_43_54": sig == ["43-54"],
                     "p_43_54": ph.set_index("bin").loc["43-54", fam]})
F1IP = pd.DataFrame(rows)
print(F1IP.to_string())
key = ("time", "both", "2-2.8", "avgmax", "all")
print("main reading, per bin:")
print(posthoc(*ANOVA[key][:2]).to_string())

**Result: not reproduced.** On the ANOVAs whose count of values is within tolerance (fig1i-baseline-contrast-interaction), the 43-54% bin is not significant under either Bonferroni family (corrected p = 0.095 to 0.22 with seven comparisons, 1 with 91), and the 54-65% bin is the only significant one under both families. Keeping every cell-bin-contrast (2204 values, outside the tolerance), the 43-54% bin is significant (LOW 15.54 against HIGH 10.43% dF/F, corrected p = 0.00017 with seven comparisons and 0.0022 with 91); with 91 comparisons it is the only one in 13 of 16 such readings, as the paper reports, while with seven the 54-65% bin is significant too.

## Figure 2: evoked PD in high and low contrast (CGC-assay trials)

Per trial, the mean evoked PD (% baseline) over 0 to 8 s and over 1 to 2 s from DRC onset; trials with
baseline PD < 50% of animal max; CTRL trials of the CGC assay. Mice readings: (a) group A, (b) group B
before ZX1, (c) all mice with CTRL CGC-assay trials (A and B pooled; they share five mice). Unit: (a)
trials, (b) per-mouse means per contrast. Two-sided Wilcoxon rank sum, high against low.

In [ ]:
ptp = tr[(tr.kind == "pt") & (tr.treat == "CTRL")]
MICE2 = {"A": GROUPS["A"], "B": GROUPS["B"], "A+B": sorted(set(GROUPS["A"]) | set(GROUPS["B"]))}
print("mice shared by A and B:", sorted(set(GROUPS["A"]) & set(GROUPS["B"])))
def fig2_test(col):
    rows = []
    for name, mice in MICE2.items():
        d = ptp[ptp.subject.isin(mice) & (ptp.base_pct_max < 50)]
        hi, lo = d[d.contrast == "HIGH"][col], d[d.contrast == "LOW"][col]
        r = stats.ranksums(hi, lo)
        pm = d.groupby(["subject", "contrast"])[col].mean().unstack()
        r2 = stats.ranksums(pm.HIGH.dropna(), pm.LOW.dropna())
        rows.append({"mice": name, "n_mice": d.subject.nunique(), "unit": "trials", "n_HIGH": len(hi), "n_LOW": len(lo),
                     "median_HIGH": hi.median().round(2), "median_LOW": lo.median().round(2),
                     "z": round(r.statistic, 3), "p": r.pvalue})
        rows.append({"mice": name, "n_mice": d.subject.nunique(), "unit": "mouse means", "n_HIGH": pm.HIGH.notna().sum(),
                     "n_LOW": pm.LOW.notna().sum(), "median_HIGH": pm.HIGH.median().round(2),
                     "median_LOW": pm.LOW.median().round(2), "z": round(r2.statistic, 3), "p": r2.pvalue})
    return pd.DataFrame(rows)
F2B = fig2_test("ev_0_8")
print(F2B.to_string())
d = ptp[ptp.subject.isin(GROUPS["A"]) & (ptp.base_pct_max < 50)]
print("group A, mean evoked PD 0-8 s by mouse and contrast (trials < 50%):")
print(d.groupby(["subject", "contrast"]).ev_0_8.agg(["mean", "count"]).unstack().round(1).to_string())
print("range of values:", d.ev_0_8.min().round(1), "to", d.ev_0_8.max().round(1))
print("CGC-assay CTRL trials of group A with a usable pupil:", int(ptp.subject.isin(GROUPS["A"]).sum()),
      "; below 50%:", len(d))
# variant without the 50% limit
dd = ptp[ptp.subject.isin(GROUPS["A"])]
r = stats.ranksums(dd[dd.contrast == "HIGH"].ev_0_8, dd[dd.contrast == "LOW"].ev_0_8)
print(f"variant, group A, all trials: z = {r.statistic:.3f}, p = {r.pvalue:.4f}")

**Result: not reproduced.** Evoked PD over 0 to 8 s is higher in HIGH than LOW under every reading, but not significantly (group A trials, medians 109.10 against 105.61%, z = 1.109, p = 0.27; group B trials p = 0.55; A and B pooled p = 0.096; per-mouse means p = 0.40 to 0.89). Only 84 of 211 group A trials have a baseline below 50% of animal max, from six mice. Without the 50% limit group A gives z = 2.551, p = 0.011 (a variant).

In [ ]:
F2C = fig2_test("ev_1_2")
print(F2C.to_string())
dd = ptp[ptp.subject.isin(GROUPS["A"])]
r = stats.ranksums(dd[dd.contrast == "HIGH"].ev_1_2, dd[dd.contrast == "LOW"].ev_1_2)
print(f"variant, group A, all trials: z = {r.statistic:.3f}, p = {r.pvalue:.4f}")

**Result: blocked (methods_underspecified).** The readings disagree: with group A and B mice pooled, trials as units, HIGH exceeds LOW (medians 109.49 against 104.47%, z = 2.581, p = 0.0098), while group A alone (p = 0.12), group B alone (p = 0.16) and every per-mouse reading (p = 0.27 to 0.63) are not significant. The criterion asks for agreement and blocks the claim otherwise. Without the 50% limit, group A gives p = 0.072.

### DRC responses of cells before and after ZX1 (Fig. 2d, e; group B)

Per cell and contrast, the mean over 0 to 2 s of the trial-averaged dF/F; tone-responsive cells under
each inclusion reading, determined within each condition (the files segment cells separately before and
after ZX1 in most mice). Two-sided Wilcoxon rank sum, low against high, in CTRL and in ZX1.

In [ ]:
ptB = trials[(trials.kind == "pt") & trials.subject.isin(GROUPS["B"]) & trials.treat.isin(["CTRL", "ZX1"])]
ctB = zp.cell_trial_table(ptB)
print("group B mice:", GROUPS["B"], "; genotypes:", summary.loc[GROUPS["B"], "genotype"].value_counts().to_dict())
print("planes by condition:", ctB.groupby("treat").plane.unique().map(list).to_dict())
rows = []
for sd in ("time", "trials"):
    for both in (True, False):
        for mice_name, mice in (("B", GROUPS["B"]), ("B without ZnT3-KO", [m for m in GROUPS["B"] if summary.loc[m, "genotype"] != "ZnT3KO"])):
            for cond in ("CTRL", "ZX1"):
                d = ctB[(ctB.treat == cond) & ctB.subject.isin(mice)]
                cells = zp.responsive(d, (2.0, 2.8), sd=sd, by_contrast=both)
                v = d[d.cell.isin(cells)].groupby(["cell", "contrast"]).drc_0_2.mean().unstack()
                r = stats.ranksums(v.LOW.dropna(), v.HIGH.dropna())
                rows.append({"SD": sd, "contrasts": "both" if both else "pooled", "mice": mice_name, "condition": cond,
                             "cells": len(cells), "median_LOW": v.LOW.median().round(3), "median_HIGH": v.HIGH.median().round(3),
                             "median_diff_LOW-HIGH": (v.LOW - v.HIGH).median().round(3), "z": round(r.statistic, 4), "p": r.pvalue})
F2E = pd.DataFrame(rows)
print(F2E.to_string())
print("paper: 520 cells from 14 mice; CTRL z = 0.0073, p = 0.9942; ZX1 z = -1.2425, p = 0.2141")

**Result: not reproduced.** DRC responses over 0 to 2 s differ between contrasts under every reading, in CTRL and after ZX1: LOW is below HIGH (median cell difference -1.19 to -1.51% dF/F; Wilcoxon rank sum z = -5.87 to -7.57, p = 3.7e-14 to 4.5e-09). Under the main inclusion reading 359 cells in CTRL and 363 after ZX1 from 12 mice, against the paper's 520 from 14; the counts are outside the tolerance under every reading (209 to 375).

## Figure 3: LME of pure-tone responses on baseline and evoked PD (group A)

Rows are cell by trial (CTRL CGC-assay trials of group A), with the trial's peak pure-tone response
(2-2.8 s window; 2-3.2 s as a variant), its baseline PD (% animal max), contrast, and the trial's mean
evoked PD over 1 to 2 s (% baseline). The Methods' random intercepts for baseline_PD and pupil_at_1-2s
group rows by their values, which are values of the trial, so each is a random intercept per trial; the
two give the same grouping and are fitted as one trial intercept nested in subject. Fitted by maximum
likelihood with statsmodels MixedLM; the fixed effects are tested with Wald z tests (statsmodels has no
Satterthwaite F; the paper's denominator df are about 165, where t and z p-values differ little).

In [ ]:
import statsmodels.regression.mixed_linear_model as mlm
def fit_fig3(d, formula):
    """ML fit; tries several optimizers and keeps the best likelihood (the fits are fragile)."""
    m = smf.mixedlm(formula, d, groups=d["subject"], re_formula="1", vc_formula={"trial": "0 + C(trial)"})
    best = None
    for method in ("lbfgs", "bfgs", "nm", "powell"):
        try:
            r = m.fit(reml=False, method=method, maxiter=2000)
        except Exception:
            continue
        if np.isfinite(r.llf) and (best is None or r.llf > best.llf):
            best = r
    return best

ct3 = ctb.copy()
# predictors centred (baseline at 50%, evoked PD at 100%) to help the optimizer; slopes are unchanged
ct3["pupil12"] = ct3.trial.map(tr.set_index("trial_id").ev_1_2) - 100
ct3["base"] = ct3["base"] - 50
ct3["HIGH"] = (ct3.contrast == "HIGH").astype(float)
rows, FITS3 = [], {}
for cells_name, cells in (("both contrasts", INCL[("time", "both")]), ("all tone-responsive", INCL[("time", "pooled")])):
    for trials_name in ("all", "<50%"):
        for wn, w in WINDOWS.items():
            d = ct3[ct3.cell.isin(cells)]
            if trials_name == "<50%":
                d = d[d.base < 0]  # base is centred at 50%
            d = d.rename(columns={f"peak_{w[0]}_{w[1]}": "peak"})[["peak", "base", "HIGH", "pupil12", "subject", "trial"]].dropna()
            full = fit_fig3(d, "peak ~ base + HIGH * pupil12")
            null = fit_fig3(d, "peak ~ base")
            FITS3[(cells_name, trials_name, wn)] = (full, null, d)
            lr = 2 * (full.llf - null.llf)
            rows.append({"cells": cells_name, "trials": trials_name, "window": wn, "rows": len(d),
                         "n_cells": d.shape[0] and ctb[ctb.cell.isin(cells)].cell.nunique(), "n_trials": d.trial.nunique(),
                         "b_base": full.params["base"], "p_base": full.pvalues["base"],
                         "b_pupil12": full.params["pupil12"], "p_pupil12": full.pvalues["pupil12"],
                         "b_HIGHxpupil12": full.params["HIGH:pupil12"], "p_HIGHxpupil12": full.pvalues["HIGH:pupil12"],
                         "b_HIGH": full.params["HIGH"], "p_HIGH": full.pvalues["HIGH"],
                         "var_subject": full.cov_re.iloc[0, 0], "var_trial": full.vcomp[0], "var_resid": full.scale,
                         "LR": lr, "LR_df": 3, "p_LRT": stats.chi2.sf(lr, 3), "converged": full.converged and null.converged})
F3 = pd.DataFrame(rows)
print(F3.to_string())
print("paper: baseline PD p = 3.64e-3; evoked PD p = 0.108; interaction p = 0.756; LRT p = 0.330")

**Result: reproduced.** In all eight fits (two cell readings, two trial readings, two peak windows) the baseline PD coefficient is negative and significant (b = -0.102 to -0.179% dF/F per % of max, p = 9.7e-05 to 0.0074), while evoked PD at 1-2 s (p = 0.13 to 0.29) and its interaction with contrast (p = 0.55 to 0.98) are not, as the paper reports. The paper does not give the sign of the baseline coefficient.

In [ ]:
print(F3[["cells", "trials", "window", "rows", "LR", "LR_df", "p_LRT", "b_HIGH", "p_HIGH"]].to_string())
print("paper: likelihood ratio test p = 0.330")

**Result: not reproduced.** The likelihood ratio test of the full model against the null model is significant in every fit (chi-squared on 3 df, p = 0.0012 to 0.048), against the paper's p = 0.330. The full model also carries the contrast main effect, which is significant when all trials are used (b = -1.57 to -1.77, p = 0.016 to 0.034) and accounts for part of the difference.

## Figure 4: growth curve analysis of evoked PD in ZnT3-WT and ZnT3-KO mice

DRC trials without the tone (`contrastChange`), CTRL and ZX1, baseline PD < 50% of animal max; pupil
(% baseline) at the 21 samples from 0 to 2 s; orthogonal time polynomials with unit-norm columns over
those samples (R `poly()`). LME with a random intercept per mouse, fitted by REML (statsmodels MixedLM;
lmer's default). Readings: model (a) the written formula plus contrast:treatment, (b) the full factorial;
coding (a) reference (LOW, CTRL), (b) sum coding (±1). Terms tested with Wald z (the paper's
Satterthwaite F have about 4500 to 7900 denominator df, where z and t agree).

In [ ]:
drc = tr[(tr.kind == "drc")].copy()
drc["is_high"] = (drc.contrast == "HIGH").astype(float)
FORMS = {"written+cxt": "pupil ~ (tPoly1 + tPoly2) * c + (tPoly1 + tPoly2) * x + c:x",
         "full factorial": "pupil ~ (tPoly1 + tPoly2) * c * x"}
def code(long, factor_col, level, coding):
    v = (long[factor_col] == level).astype(float)
    return v if coding == "reference" else 2 * v - 1

def gca(trs, factor="treat", level="ZX1", forms=FORMS, codings=("reference", "sum")):
    long = zp.gca_long(trs)
    long["c"] = 0.0
    out, fits = [], {}
    for coding in codings:
        long["c"] = code(long, "contrast", "HIGH", coding)
        long["x"] = code(long, factor, level, coding)
        for fname, f in forms.items():
            m = smf.mixedlm(f, long, groups=long["subject"]).fit(reml=True, method=["lbfgs", "bfgs"])
            fits[(fname, coding)] = (m, long.copy())
            row = {"model": fname, "coding": coding, "rows": len(long), "trials": long.trial.nunique(),
                   "mice": long.subject.nunique(), "converged": m.converged}
            for term, lab in (("c", "contrast_int"), ("tPoly1:c", "contrast_slope"), ("x", "x_int"),
                              ("tPoly1:x", "x_slope"), ("c:x", "c_x")):
                if term in m.params:
                    row[f"{lab}_b"] = m.params[term]; row[f"{lab}_z"] = m.tvalues[term]; row[f"{lab}_p"] = m.pvalues[term]
            out.append(row)
    return pd.DataFrame(out), fits, long

C, D = GROUPS["C"], GROUPS["D"]
wt = drc[drc.subject.isin(C) & drc.treat.isin(["CTRL", "ZX1"]) & (drc.base_pct_max < 50)]
print("ZnT3-WT DRC trials with a usable pupil:", int((drc.subject.isin(C)).sum()), "; below 50%:", len(wt))
print(pd.crosstab([wt.subject], [wt.treat, wt.contrast]).to_string())
F4C, FITS_WT, long_wt = gca(wt)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
print(F4C.to_string())
print("paper (WT): contrast intercept F = 19.48 p = 1.03e-5; contrast slope F = 26.54 p = 2.64e-7;"
      " ZX1 intercept F = 4.47 p = 0.0344; ZX1 slope F = 40.45 p = 2.13e-10; contrast x ZX1 F = 24.2 p = 8.85e-7; df about 7920")
terms = ["contrast_int_p", "contrast_slope_p", "x_int_p", "x_slope_p", "c_x_p"]
print("terms with p < 0.05 per reading:", (F4C[terms] < 0.05).sum(1).tolist())

In [ ]:
# mean traces (0-2 s) by condition, and per-mouse mean evoked PD over 0-2 s
mt = long_wt.groupby(["treat", "contrast", "time"]).pupil.mean().unstack(["treat", "contrast"])
print(mt.iloc[::4].round(2).to_string())
print(wt.groupby(["subject", "treat", "contrast"]).ev_0_2.mean().unstack(["treat", "contrast"]).round(1).to_string())
fig, ax = plt.subplots(figsize=(4, 3))
for c in mt.columns:
    ax.plot(mt.index, mt[c], label=" ".join(c))
ax.legend(); ax.set(xlabel="time (s)", ylabel="pupil (% baseline)", title="ZnT3-WT"); plt.tight_layout(); plt.show()
print("plotted range", np.nanmin(mt.values).round(1), "to", np.nanmax(mt.values).round(1))

**Result: not reproduced.** 361 ZnT3-WT trials below 50% (7581 rows, close to the paper's denominator df of about 7920). Of the five terms, the contrast intercept is significant under reference coding (b = 1.018, z = 2.39, p = 0.017; under sum coding p = 0.057), the contrast slope under the full factorial with reference coding only (p = 0.036), the ZX1 intercept under none (p >= 0.47), the ZX1 slope under all four readings (z = -2.94 to -5.14, p <= 0.0033), and the contrast by ZX1 interaction under none (z = -1.15, p = 0.25). At most three of five terms are significant and never the interaction, so no reading meets the full or the partial criterion. The mean traces show the expected order in CTRL (HIGH 111.9 against LOW 109.1% at 2 s) and a smaller, reversed one after ZX1 (107.5 against 108.4).

In [ ]:
def emmeans_tukey(m, long, coding):
    """Cell means at tPoly1 = tPoly2 = 0 for the 4 contrast x treatment cells, Tukey-adjusted pairs."""
    names = m.fe_params.index
    cov = m.cov_params().loc[names, names].values
    lv = {"reference": (0.0, 1.0), "sum": (-1.0, 1.0)}[coding]
    cells = {}
    for cn, cv in (("LOW", lv[0]), ("HIGH", lv[1])):
        for xn, xv in (("CTRL", lv[0]), ("X", lv[1])):
            L = np.zeros(len(names))
            for i, n in enumerate(names):
                if n == "Intercept": L[i] = 1
                elif n == "c": L[i] = cv
                elif n == "x": L[i] = xv
                elif n == "c:x": L[i] = cv * xv
            cells[(cn, xn)] = L
    df_ = len(long) - len(names)
    rows = []
    keys = list(cells)
    for i in range(4):
        for j in range(i + 1, 4):
            L = cells[keys[i]] - cells[keys[j]]
            est = L @ m.fe_params.values
            se = np.sqrt(L @ cov @ L)
            tval = est / se
            p = stats.studentized_range.sf(abs(tval) * np.sqrt(2), 4, df_)
            rows.append({"pair": f"{keys[i][0]} {keys[i][1]} - {keys[j][0]} {keys[j][1]}", "estimate": est, "z": tval, "p_tukey": p})
    return pd.DataFrame(rows), {k: v @ m.fe_params.values for k, v in cells.items()}

def posthoc_summary(fits):
    rows = []
    for key, (m, long) in fits.items():
        ph, means = emmeans_tukey(m, long, key[1])
        p = ph.set_index("pair").p_tukey; e = ph.set_index("pair").estimate
        rows.append({"model": key[0], "coding": key[1],
                     "HIGH-LOW CTRL": e["LOW CTRL - HIGH CTRL"] * -1, "p": p["LOW CTRL - HIGH CTRL"],
                     "HIGH-LOW X": e["LOW X - HIGH X"] * -1, "p ": p["LOW X - HIGH X"],
                     "X-CTRL HIGH": e["HIGH CTRL - HIGH X"] * -1, "p  ": p["HIGH CTRL - HIGH X"],
                     "X-CTRL LOW": e["LOW CTRL - LOW X"] * -1, "p   ": p["LOW CTRL - LOW X"],
                     **{f"mean {k[0]} {k[1]}": v for k, v in means.items()}})
    return pd.DataFrame(rows)
PH_WT = posthoc_summary(FITS_WT)
print(PH_WT.to_string())
ok = ((PH_WT["HIGH-LOW CTRL"] > 0) & (PH_WT["p"] < 0.05) & ~(PH_WT["p "] < 0.05) & (PH_WT["X-CTRL HIGH"] < 0)
      & (PH_WT["p  "] < 0.05) & ~(PH_WT["p   "] < 0.05))
print("criterion met per reading:", ok.tolist())
print(emmeans_tukey(*FITS_WT[("written+cxt", "reference")], "reference")[0].to_string())

**Result: not reproduced.** At the cell means, HIGH exceeds LOW in CTRL by 1.018% (Tukey p = 0.079), not significant, and ZX1 does not lower HIGH (-0.418, p = 0.84) or LOW (0.346, p = 0.89). The same numbers come out under all four readings.

In [ ]:
ko = drc[drc.subject.isin(D) & drc.treat.isin(["CTRL", "ZX1"]) & (drc.base_pct_max < 50)]
print("ZnT3-KO DRC trials with a usable pupil:", int((drc.subject.isin(D)).sum()), "; below 50%:", len(ko))
print(pd.crosstab([ko.subject], [ko.treat, ko.contrast]).to_string())
F4F, FITS_KO, long_ko = gca(ko)
print(F4F.to_string())
print("paper (KO): contrast intercept p = 0.255; ZX1 intercept p = 0.147; contrast x ZX1 p = 0.589;"
      " contrast slope p = 0.190; ZX1 slope p = 0.0129 (weight -0.64); df about 4500")
absent = ["contrast_int_p", "x_int_p", "c_x_p", "contrast_slope_p"]
print("absence terms all p >= 0.05 per reading:", (F4F[absent] >= 0.05).all(1).tolist())
mt = long_ko.groupby(["treat", "contrast", "time"]).pupil.mean().unstack(["treat", "contrast"])
print(mt.iloc[::4].round(2).to_string())

**Result: blocked (methods_underspecified).** 181 ZnT3-KO trials below 50% (3801 rows, against about 4500 in the paper). The contrast intercept (p = 0.39 to 0.58), ZX1 intercept (p = 0.18 to 0.30) and contrast by ZX1 (p = 0.91) terms are not significant under any reading, but the contrast slope is significant in three of four readings (z = 2.63 to 2.74, p = 0.0062 to 0.0086) and not under the full factorial with reference coding (p = 0.245). The criterion treats a disagreement between model readings as methods_underspecified.

In [ ]:
cmp = pd.DataFrame({"model": F4F.model, "coding": F4F.coding,
                    "KO ZX1 slope b": F4F.x_slope_b, "KO p": F4F.x_slope_p, "WT ZX1 slope b": F4C.x_slope_b, "WT p": F4C.x_slope_p})
cmp["KO < half WT"] = cmp["KO ZX1 slope b"].abs() < 0.5 * cmp["WT ZX1 slope b"].abs()
cmp["KO in -0.64±0.064"] = (cmp["KO ZX1 slope b"] + 0.64).abs() <= 0.064
cmp["WT in -1.41±0.141"] = (cmp["WT ZX1 slope b"] + 1.41).abs() <= 0.141
print(cmp.to_string())

**Result: blocked (methods_underspecified).** This claim depends on fig4f-ko-no-contrast-effect, which is blocked, so it is blocked for the same reason. The analysis ran: the ZX1 slope term in ZnT3-KO mice is negative but not significant under any reading (b = -0.547 to -3.133, p = 0.20 to 0.52). The KO weight is less than half the WT weight in magnitude in three of four readings (for example -1.095 against -7.684, written model with reference coding). Neither weight is near the expected values (-0.64 and -1.41): with unit-norm polynomials over the 21 samples the WT weight is -3.84 to -7.68, so the paper's polynomial scaling is not this one.

## Figure 5

The three Figure 5 claims (fig5-contralateral-zx1-increases, fig5-ipsilateral-zx1-reduces,
fig5c-no-dye-in-other-hemisphere) need the widefield imaging of right A1 in three mice with a cannula in
the left auditory cortex, before and after contralateral and ipsilateral ZX1, and the red-channel images.
All 30 files hold two-photon imaging and pupillometry (above); no file holds widefield imaging or
cannula experiments, so these claims are blocked (data_missing).

## Figure 6: baseline PD drift, ACSF control, baseline against evoked PD, and genotype

### Baseline PD against recording time (Fig. 6a)

Per trial (CGC-assay and DRC trials, all treatments), baseline PD in % animal max against the time of
sound onset in minutes from the experiment's first trial; ordinary least squares pooled over mice. ZX1
readings: (a) groups C and D, (b) group C, (c) groups B, C and D, and (d) the eleven mice the files label
`ZX1_uBasePupilPctMax_wEpoch`. ACSF: group F (the files' `ACSF_uBasePupilPctMax_wEpoch` label names the
same four mice).

In [ ]:
snd_tr = tr[tr.kind.isin(["pt", "drc"])]
def drift(mice, label):
    d = snd_tr[snd_tr.subject.isin(mice)]
    r = stats.linregress(d.rec_min, d.base_pct_max)
    tcrit = stats.t.ppf(0.975, len(d) - 2)
    return {"set": label, "mice": d.subject.nunique(), "trials": len(d), "slope": r.slope,
            "ci_low": r.slope - tcrit * r.stderr, "ci_high": r.slope + tcrit * r.stderr, "p": r.pvalue,
            "minutes": f"{d.rec_min.min():.0f} to {d.rec_min.max():.0f}"}
rows = [drift(C + D, "ZX1 (a) C+D"), drift(C, "ZX1 (b) C"), drift(sorted(set(GROUPS["B"]) | set(C) | set(D)), "ZX1 (c) B+C+D"),
        drift(GROUPS["ZX1_drift"], "ZX1 (d) labelled"), drift(GROUPS["F"], "ACSF F")]
F6A = pd.DataFrame(rows)
print(F6A.to_string())
acsf = F6A.iloc[-1]
for _, r in F6A.iloc[:-1].iterrows():
    print(f"{r['set']}: slope > 0 and p < 0.05: {r.slope > 0 and r.p < 0.05}; CI overlaps ACSF CI: "
          f"{max(r.ci_low, acsf.ci_low) <= min(r.ci_high, acsf.ci_high)}")
print("ACSF: slope > 0 and p < 0.05:", acsf.slope > 0 and acsf.p < 0.05)
print("paper: ZX1 CI (0.0798, 0.183), p = 1.26e-6; ACSF CI (0.0159, 0.279), p = 0.0282")
print("per-mouse slopes (% per min):", snd_tr.groupby("subject").apply(lambda g: stats.linregress(g.rec_min, g.base_pct_max).slope).round(3).to_dict())

**Result: not reproduced.** Baseline PD rises with recording time in both experiments: under the four ZX1 readings slopes of 0.123 to 0.194% of max per minute (95% CIs within 0.098 to 0.225, p <= 2.8e-15), and in the ACSF mice 0.326 (CI 0.281 to 0.371, p = 1.9e-34). But the ACSF interval does not overlap any ZX1 interval, which the criterion requires; the paper's ACSF interval (0.0159 to 0.279) is wider and lower. Per-mouse slopes range from -0.349 to 0.776.

### Contrast effect after ACSF (Fig. 6b)

Group F, DRC trials, baseline PD < 50%. Readings: (a) ACSF trials only, pupil ~ (tPoly1 + tPoly2) x
contrast + (1 | subject); (b) trials before and after ACSF, the models of Fig. 4 with ACSF as treatment,
and the contrast effect within ACSF from the cell means.

In [ ]:
F_ = GROUPS["F"]
ac = drc[drc.subject.isin(F_) & (drc.base_pct_max < 50)]
print("group F DRC trials with a usable pupil:", int(drc.subject.isin(F_).sum()), "; below 50%:", len(ac))
print(pd.crosstab([ac.subject], [ac.treat, ac.contrast]).to_string())
ra, _, long_a = gca(ac[ac.treat == "ACSF"], forms={"post only": "pupil ~ (tPoly1 + tPoly2) * c"})
print("(a) ACSF trials only")
print(ra[["model", "coding", "rows", "trials", "mice", "contrast_int_b", "contrast_int_p", "contrast_slope_b", "contrast_slope_p"]].to_string())
rb, fits_b, long_b = gca(ac[ac.treat.isin(["CTRL", "ACSF"])], factor="treat", level="ACSF")
print("(b) before and after ACSF")
print(rb.to_string())
phb = posthoc_summary(fits_b)
print(phb[["model", "coding", "HIGH-LOW CTRL", "p", "HIGH-LOW X", "p "]].rename(columns={"HIGH-LOW X": "HIGH-LOW ACSF", "p ": "p ACSF"}).to_string())
print("paper: contrast intercept F(1,2558.3) = 46.22 p = 1.31e-11; slope F(1,2558) = 15.076 p = 1.10e-4")
mt = long_b.groupby(["treat", "contrast", "time"]).pupil.mean().unstack(["treat", "contrast"])
print(mt.iloc[::4].round(2).to_string())

**Result: partially reproduced.** The contrast intercept effect holds after ACSF under every reading (ACSF trials only, b = 4.05% with HIGH above LOW, p = 2.8e-05; trials before and after, contrast term p = 0.00056 to 0.0015), but the contrast slope does not (p = 0.070 with ACSF trials only, 0.27 to 0.83 with both). With trials before and after, the HIGH minus LOW difference at the cell means is 3.97 in CTRL (Tukey p = 0.0032) and 3.03 after ACSF (p = 0.37). Only 35 ACSF trials (735 rows, against about 2560 in the paper) are below 50%.

### Baseline PD against evoked PD (Fig. 6c, and the correlation in the Methods)

Per trial, baseline PD (% animal max) against the mean evoked PD over 1 to 2 s (% baseline), both
contrasts. Every reading of the claims file: groups A, B (CGC-assay trials), C, D, F (DRC trials) and the
files' `pupilBaseline_vs_evoked` label (DRC, CGC-assay or both kinds), CTRL trials or all trials.

In [ ]:
rows = []
kinds = {"A": ["pt"], "B": ["pt"], "C": ["drc"], "D": ["drc"], "F": ["drc"], "base_vs_evoked": ["drc", "pt"]}
for g, ks in kinds.items():
    for kind in ks + (["pt", "drc"] if g == "base_vs_evoked" else []):
        kk = [kind] if isinstance(kind, str) else kind
    for kind in ([ks] if len(ks) == 1 else [["drc"], ["pt"], ["drc", "pt"]]):
        for trt in ("CTRL", "all"):
            d = snd_tr[snd_tr.subject.isin(GROUPS[g]) & snd_tr.kind.isin(kind)]
            if trt == "CTRL":
                d = d[d.treat == "CTRL"]
            r_all = stats.pearsonr(d.base_pct_max, d.ev_1_2)
            d50 = d[d.base_pct_max < 50]
            r50 = stats.pearsonr(d50.base_pct_max, d50.ev_1_2)
            rows.append({"group": g, "kind": "+".join(kind), "trials": trt, "n_all": len(d), "r_all": r_all.statistic,
                         "p_all": r_all.pvalue, "n_below50": len(d50), "r_below50": r50.statistic, "p_below50": r50.pvalue})
F6C = pd.DataFrame(rows)
F6C["n50_matches_240"] = (F6C.n_below50 - 240).abs() <= 24
print(F6C.to_string())
print("paper (Fig. 6c): r(238) = -0.0928, p = 0.1537, 240 responses")

**Result: reproduced.** Of the claims file's readings (groups A, B, C, D, F; CTRL or all trials), one has a count within tolerance of 240 below 50%: group B, CGC-assay trials before and after ZX1 (262 responses, r = -0.111, p = 0.072), not significant, as the paper reports (r = -0.0928, p = 0.1537). The files' `pupilBaseline_vs_evoked` mice, DRC CTRL trials, also match the count (222 responses, r = 0.030, p = 0.66) and agree. Over the other readings, below-50% correlations range from -0.270 to 0.067, and 5 of 16 are significant.

In [ ]:
F6C["n_matches_626"] = (F6C.n_all - 626).abs() <= 62.6
F6C["n50_matches_384"] = (F6C.n_below50 - 384).abs() <= 38.4
print(F6C[["group", "kind", "trials", "n_all", "r_all", "p_all", "n_below50", "r_below50", "p_below50", "n_matches_626", "n50_matches_384"]].to_string())
print("paper (Methods): r(624) = -0.35, p = 1.20e-19 over 626 responses; r(382) = -0.055, p = 0.285 below 50%")
# Variants: evoked PD over 0-2 s and 0-8 s, for the readings whose counts match
for _, r in F6C[F6C.n_matches_626 & F6C.n50_matches_384].iterrows():
    d = snd_tr[snd_tr.subject.isin(GROUPS[r.group]) & snd_tr.kind.isin(r.kind.split("+"))]
    if r.trials == "CTRL":
        d = d[d.treat == "CTRL"]
    for col in ("ev_0_2", "ev_0_8"):
        a = stats.pearsonr(d.base_pct_max, d[col]); b = stats.pearsonr(d[d.base_pct_max < 50].base_pct_max, d[d.base_pct_max < 50][col])
        print(f"variant {r.group} {r.kind} {r.trials} {col}: all r = {a.statistic:.3f} p = {a.pvalue:.3g}; below 50% r = {b.statistic:.3f} p = {b.pvalue:.3g}")

**Result: partially reproduced (resting on an assumption).** None of the claims file's readings (groups A, B, C, D, F; CTRL or all trials) matches both counts (the closest, group C all trials, has 501 responses and 361 below 50%), and they do not agree with each other: over all responses r ranges from -0.215 to -0.422, inside the tolerance (-0.315 to -0.385) for group A and for group B CTRL only. The files' `pupilBaseline_vs_evoked` label names the mice of this analysis, and its DRC trials, CTRL and ZX1, match both counts (598 responses, within 626 ± 62.6; 379 below 50%, within 384 ± 38.4); this reading is adopted as an assumption. On it, the correlation over all responses is negative and significant (r = -0.218, p = 7.4e-08) but weaker than the paper's -0.35 and outside its tolerance, and below 50% it is not significant (r = 0.067, p = 0.20), as the paper reports. With evoked PD over 0-2 s or 0-8 s the picture is the same (r = -0.200 and -0.217 over all; p = 0.092 and 0.82 below 50%).

### Genotype effect on high-contrast responses before ZX1 (Fig. 6d)

In [ ]:
gd = drc[drc.subject.isin(C + D) & (drc.treat == "CTRL") & (drc.contrast == "HIGH") & (drc.base_pct_max < 50)].copy()
gd["genotype"] = gd.subject.map(summary.genotype)
r6d, fits6d, long6d = gca(gd, factor="genotype", level="ZnT3KO", forms={"genotype": "pupil ~ (tPoly1 + tPoly2) * x"})
print(r6d[["model", "coding", "rows", "trials", "mice", "x_int_b", "x_int_p", "x_slope_b", "x_slope_z", "x_slope_p"]].to_string())
print("paper: genotype on slope F(1,3869) = 22.37, p = 2.33e-6 (about 3870 rows)")
mt = long6d.groupby(["genotype", "time"]).pupil.mean().unstack("genotype")
print(mt.iloc[::4].round(2).to_string())

**Result: reproduced.** 144 CTRL high-contrast trials of 16 ZnT3-WT and ZnT3-KO mice (3024 rows, against about 3870). The genotype by tPoly1 term is negative, so the KO slope is below the WT slope (b = -6.50 with reference coding, z = -2.73, p = 0.0063; the same test under sum coding). The genotype intercept term is not significant (p = 0.58). The mean KO trace rises faster at first and is below WT from 1.2 s (110.7 against 111.5 at 1.2 s, 109.4 against 111.9 at 2 s).

### Peak of high-contrast responses (Discussion; Fig. 4c, d)

In [ ]:
def peak_rise(trs):
    long = zp.gca_long(trs)
    m = long.groupby("time").pupil.mean()
    return m.max() - 100, m.idxmax(), long.trial.nunique()
rows = []
for name, d in (("WT after ZX1, HIGH", wt[(wt.treat == "ZX1") & (wt.contrast == "HIGH")]),
                ("WT CTRL, HIGH", wt[(wt.treat == "CTRL") & (wt.contrast == "HIGH")]),
                ("KO CTRL, HIGH (a)", ko[(ko.treat == "CTRL") & (ko.contrast == "HIGH")]),
                ("KO CTRL+ZX1, HIGH (b)", ko[ko.contrast == "HIGH"]),
                ("WT after ZX1, LOW", wt[(wt.treat == "ZX1") & (wt.contrast == "LOW")]),
                ("WT CTRL, LOW", wt[(wt.treat == "CTRL") & (wt.contrast == "LOW")])):
    pk, tpk, n = peak_rise(d)
    rows.append({"set": name, "trials": n, "peak_rise_%": round(pk, 2), "at_s": tpk, "within_10±2.5": abs(pk - 10) <= 2.5})
print(pd.DataFrame(rows).to_string())

**Result: reproduced.** The mean high-contrast trace of ZnT3-WT mice after ZX1 peaks 10.42% above baseline (70 trials), and that of ZnT3-KO mice 10.84% in CTRL (49 trials) and 10.83% with CTRL and ZX1 pooled, all within 10 ± 2.5. The measure does not separate conditions well: the WT CTRL high-contrast peak (11.94%) and the WT low-contrast peaks (9.38 and 10.63%) are inside the same interval. The second part of the statement, that both contrasts resemble low-contrast responses, is not tested by the criterion; fig4-wt-posthoc covers it for WT and was not reproduced.

## Variant: the paper's single cleaning pass

The pupil claims rest on the cleaning assumption (Data check). Here the main pupil numbers are recomputed
with the Methods' single pass (no spike filter, no trial dropped for missing frames), where several
animals' "maximum" PD is a tracking failure.

In [ ]:
sv = single.copy()
sv_drc = sv[sv.kind == "drc"]
out = []
for name, mice in MICE2.items():
    d = sv[(sv.kind == "pt") & (sv.treat == "CTRL") & sv.subject.isin(mice) & (sv.base_pct_max < 50)]
    for col in ("ev_0_8", "ev_1_2"):
        r = stats.ranksums(d[d.contrast == "HIGH"][col], d[d.contrast == "LOW"][col])
        out.append({"analysis": f"Fig 2 {col} {name} trials", "n": len(d), "stat": round(r.statistic, 3), "p": r.pvalue})
wts = sv_drc[sv_drc.subject.isin(C) & sv_drc.treat.isin(["CTRL", "ZX1"]) & (sv_drc.base_pct_max < 50)]
r4, _, _ = gca(wts, forms={"written+cxt": FORMS["written+cxt"]}, codings=("reference",))
for t_ in ("contrast_int", "contrast_slope", "x_int", "x_slope", "c_x"):
    out.append({"analysis": f"Fig 4c WT {t_}", "n": int(r4.rows[0]), "stat": r4[f"{t_}_z"][0], "p": r4[f"{t_}_p"][0]})
kos = sv_drc[sv_drc.subject.isin(D) & sv_drc.treat.isin(["CTRL", "ZX1"]) & (sv_drc.base_pct_max < 50)]
r4k, _, _ = gca(kos, forms={"written+cxt": FORMS["written+cxt"]}, codings=("reference",))
for t_ in ("contrast_int", "contrast_slope", "x_int", "x_slope", "c_x"):
    out.append({"analysis": f"Fig 4f KO {t_}", "n": int(r4k.rows[0]), "stat": r4k[f"{t_}_z"][0], "p": r4k[f"{t_}_p"][0]})
acs = sv_drc[sv_drc.subject.isin(F_) & (sv_drc.treat == "ACSF") & (sv_drc.base_pct_max < 50)]
r6, _, _ = gca(acs, forms={"post only": "pupil ~ (tPoly1 + tPoly2) * c"}, codings=("reference",))
for t_ in ("contrast_int", "contrast_slope"):
    out.append({"analysis": f"Fig 6b ACSF {t_}", "n": int(r6.rows[0]), "stat": r6[f"{t_}_z"][0], "p": r6[f"{t_}_p"][0]})
print(pd.DataFrame(out).to_string())
print("trials below 50% of animal max, single pass:", int((sv.base_pct_max < 50).sum()), "of", len(sv),
      "; main cleaning:", int((tr.base_pct_max < 50).sum()), "of", len(tr))

## Summary

| Claim | Outcome |
|---|---|
| fig1f-tone-response-inverse-u | reproduced |
| fig1g-reliability-intermediate | reproduced |
| fig1h-cgc-largest-intermediate | reproduced |
| fig1h-cgc-evident-all-bins | not reproduced |
| fig1i-baseline-contrast-interaction | reproduced |
| fig1i-contrast-effect-intermediate-only | not reproduced |
| fig2b-evoked-pd-whole-drc | not reproduced |
| fig2c-evoked-pd-1-2s | blocked: methods underspecified |
| fig2e-drc-responses-no-difference | not reproduced |
| fig3b-lme-coefficients | reproduced |
| fig3c-lme-likelihood-ratio | not reproduced |
| fig4c-wt-contrast-zx1-effects | not reproduced |
| fig4-wt-posthoc | not reproduced |
| fig4f-ko-no-contrast-effect | blocked: methods underspecified |
| fig4f-ko-zx1-slope | blocked: methods underspecified |
| fig5-contralateral-zx1-increases | blocked: data missing |
| fig5-ipsilateral-zx1-reduces | blocked: data missing |
| fig5c-no-dye-in-other-hemisphere | blocked: data missing |
| fig6a-baseline-pd-drift | not reproduced |
| fig6b-acsf-contrast-effect | partially reproduced |
| fig6c-no-baseline-evoked-correlation | reproduced |
| methods-baseline-evoked-correlation | partially reproduced |
| fig6d-ko-reduced-high-contrast | reproduced |
| discussion-zx1-wt-like-ko | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's:

- The files hold the pupil radius per video frame, with dropped frames and tracking failures (up to 1e6
  pixels), not the DeepLabCut points. The Methods' single derivative pass leaves these failures in, so a
  spike filter was added, the pass repeated, and trials with more than half their frames missing dropped.
  This changes each animal's maximum PD, hence the baseline in % of maximum and the 50% selection. The
  single pass is shown as a variant (written model, reference coding only): it leaves the Fig. 2, Fig. 4c
  and Fig. 6b results as they are, but in ZnT3-KO mice it makes the contrast intercept (p = 8.9e-06) and
  contrast by ZX1 (p = 0.040) terms significant, which would make fig4f-ko-no-contrast-effect not
  reproduced in place of blocked.
- The DRC trials without the tone switch contrast after 10 s; a trial's contrast is that of its first
  10 s, which covers the analysed 0 to 2 s.
- Groups are taken from the files' analysis labels; group A holds ten mice (paper eight), group B twelve
  (paper fourteen), and the ZnT3-WT group five F and five M (paper six F and four M).
- LME tests are Wald z tests from statsmodels, not Satterthwaite F tests; the Fig. 3 random intercepts
  for baseline and evoked PD are fitted as one trial intercept.
- Recording time for Fig. 6a comes from the .tif creation times, since the table's starting times wrap
  by one day for trials recorded before the reference file.

What was found about the dataset: the files carry labels naming the paper's analyses, a per-trial table
of sounds and treatments, ROI fluorescence and pupil radius, enough to test 21 of 24 claims. The
widefield experiments of Figure 5 are not in it. Pupil traces hold tracking failures; 25 pupil series
have a length that does not match their trial; every pupil series's description names the wrong
TwoPhotonSeries; the stimulus table's starting times wrap by 24 h for 84 trials; the paper's
contrast-switching DRC stimulus is not described in the paper; and group sizes differ from the paper.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min")